# MLFX Research Walkthrough

This notebook explores an MLFX experiment from labelled market data through prediction outputs and backtest artifacts. It is artifact-aware: optional sections are skipped cleanly when predictions, trade reports, or run logs are not available yet.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json

import numpy as np
import pandas as pd
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from mlfx.config.paths import ProjectPaths
from mlfx.training.data import load_labelled_dataset
from mlfx.training.feature_selection import select_numeric_feature_columns
from mlfx.evaluation.reporting import (
    get_candlestick_figure,
    get_drawdown_analysis_figure,
    get_equity_curve_figure,
    get_trade_distribution_figure,
)

pd.options.display.max_columns = 120
px.defaults.template = "plotly_dark"

SYMBOL = "XAUUSD"
TIMEFRAME = "1H"
LABEL = "label_10"
TRAIN_START = None
TRAIN_END = None
MAX_FEATURES_FOR_HEATMAP = 20


In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "config.toml").exists() and (candidate / "mlfx").exists():
            return candidate
    raise FileNotFoundError("Could not locate the MLFX repository root from the current working directory.")


REPO_ROOT = find_repo_root()
PATHS = ProjectPaths(project_root=REPO_ROOT)


def latest_prediction_path(paths: ProjectPaths, symbol: str, tf: str, label: str) -> Path | None:
    candidate = paths.predictions_label_dir(symbol, tf, label) / "predictions.parquet"
    return candidate if candidate.exists() else None


def latest_trade_report_path(paths: ProjectPaths, symbol: str, tf: str, label: str) -> Path | None:
    base_dir = paths.reports_label_dir(symbol, tf, label)
    if not base_dir.exists():
        return None
    trade_files = sorted(base_dir.rglob("*_trades.parquet"), key=lambda p: p.stat().st_mtime, reverse=True)
    return trade_files[0] if trade_files else None


def metrics_log_path(paths: ProjectPaths, symbol: str, tf: str, label: str) -> Path:
    return paths.runs_label_dir(symbol, tf, label) / "metrics_log.jsonl"


def compact_summary(df: pl.DataFrame, label: str) -> pd.DataFrame:
    feature_cols = select_numeric_feature_columns(df)
    summary = {
        "rows": [df.height],
        "columns": [df.width],
        "feature_columns": [len(feature_cols)],
        "timestamp_min": [df["timestamp"].min() if "timestamp" in df.columns else None],
        "timestamp_max": [df["timestamp"].max() if "timestamp" in df.columns else None],
        "null_cells": [int(df.null_count().select(pl.sum_horizontal(pl.all())).item())],
        "label_present": [label in df.columns],
    }
    return pd.DataFrame(summary)


def show_or_note(fig: go.Figure | None, message: str) -> None:
    if fig is None:
        print(message)
    else:
        fig.show()


print(f"Repo root: {REPO_ROOT}")
print(f"Symbol/timeframe/label: {SYMBOL} / {TIMEFRAME} / {LABEL}")


## 1. Load Labelled Dataset

This section reads the canonical labelled dataset that MLFX uses for training and downstream evaluation.

In [ ]:
dataset = load_labelled_dataset(
    SYMBOL,
    TIMEFRAME,
    paths=PATHS,
    train_start=TRAIN_START,
    train_end=TRAIN_END,
)

if dataset is None or dataset.is_empty():
    raise RuntimeError(
        f"No labelled dataset found for {SYMBOL}/{TIMEFRAME}. Run the MLFX pipeline first."
    )

feature_cols = select_numeric_feature_columns(dataset)
print(compact_summary(dataset, LABEL))
dataset.head(5)


## 2. Market Overview

Inspect the market structure and selected indicators on top of the underlying time series.

In [ ]:
market_fig = None
if {"timestamp", "open", "high", "low", "close"}.issubset(dataset.columns):
    market_fig = get_candlestick_figure(
        dataset.tail(min(500, dataset.height)),
        title=f"Market Overview: {SYMBOL} {TIMEFRAME} ({LABEL})",
    )

show_or_note(market_fig, "Skipping candlestick view because OHLC columns are not available.")


In [ ]:
indicator_candidates = [
    "close",
    "rsi_14",
    "ema_20",
    "ema_50",
    "ema_200",
    "atr_14",
    "macd",
    "macd_signal",
]
available = [name for name in indicator_candidates if name in dataset.columns]

if "timestamp" in dataset.columns and len(available) >= 2:
    tail_pd = dataset.select(["timestamp", *available]).tail(min(500, dataset.height)).to_pandas()
    fig = px.line(
        tail_pd,
        x="timestamp",
        y=available,
        title=f"Price and Indicator Snapshot: {SYMBOL} {TIMEFRAME}",
    )
    fig.update_layout(legend_title_text="Series")
    fig.show()
else:
    print("Skipping indicator line chart because the expected columns are not available.")


## 3. Feature and Label Exploration

Review class balance, feature relationships, and a few representative feature distributions.

In [ ]:
if LABEL in dataset.columns:
    label_counts = (
        dataset.group_by(LABEL)
        .len()
        .sort(LABEL)
        .rename({"len": "count"})
        .to_pandas()
    )
    fig = px.bar(label_counts, x=LABEL, y="count", title=f"Label Balance: {LABEL}")
    fig.show()
    label_counts
else:
    print(f"Label column {LABEL!r} is not present in the dataset.")


In [ ]:
heatmap_features = feature_cols[:MAX_FEATURES_FOR_HEATMAP]
if len(heatmap_features) >= 2:
    corr_pd = dataset.select(heatmap_features).to_pandas().corr(numeric_only=True)
    fig = px.imshow(
        corr_pd,
        title=f"Feature Correlation Heatmap (first {len(heatmap_features)} features)",
        color_continuous_scale="RdBu",
        zmin=-1,
        zmax=1,
        aspect="auto",
    )
    fig.show()
else:
    print("Not enough numeric feature columns to render a correlation heatmap.")


In [ ]:
distribution_features = feature_cols[: min(6, len(feature_cols))]
if distribution_features:
    melted = dataset.select(distribution_features).to_pandas().melt(var_name="feature", value_name="value")
    fig = px.histogram(
        melted,
        x="value",
        facet_col="feature",
        facet_col_wrap=2,
        nbins=40,
        title="Feature Distributions (sample)",
    )
    fig.update_layout(showlegend=False, height=900)
    fig.show()
else:
    print("No feature columns available for distribution plots.")


## 4. Prediction Review

If batch predictions have been generated, compare their distribution with the labels and overlay signals on price.

In [ ]:
prediction_path = latest_prediction_path(PATHS, SYMBOL, TIMEFRAME, LABEL)
predictions_df = pl.read_parquet(prediction_path) if prediction_path else None

print(f"Prediction artifact: {prediction_path}")
if predictions_df is None:
    print("Prediction section skipped. Run `pixi run mlfx batch --symbol ... --tf ... --label ...` first.")
else:
    display(predictions_df.head(5).to_pandas())

    if LABEL in predictions_df.columns and "prediction" in predictions_df.columns:
        compare_df = predictions_df.select([LABEL, "prediction"]).to_pandas()
        dist_df = pd.concat(
            [
                compare_df[LABEL].value_counts().rename("count").rename_axis("value").reset_index().assign(series="label"),
                compare_df["prediction"].value_counts().rename("count").rename_axis("value").reset_index().assign(series="prediction"),
            ],
            ignore_index=True,
        )
        fig = px.bar(dist_df, x="value", y="count", color="series", barmode="group", title="Label vs Prediction Distribution")
        fig.show()

        confusion = pd.crosstab(compare_df[LABEL], compare_df["prediction"], rownames=["label"], colnames=["prediction"])
        display(confusion)

    if {"timestamp", "close", "prediction"}.issubset(predictions_df.columns):
        tail_pd = predictions_df.select(["timestamp", "close", "prediction"]).tail(min(500, predictions_df.height)).to_pandas()
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=tail_pd["timestamp"], y=tail_pd["close"], mode="lines", name="close"))
        fig.add_trace(
            go.Scatter(
                x=tail_pd["timestamp"],
                y=tail_pd["close"],
                mode="markers",
                marker=dict(
                    size=7,
                    color=tail_pd["prediction"],
                    colorscale="RdYlGn",
                    showscale=True,
                    colorbar=dict(title="prediction"),
                ),
                name="prediction signal",
            )
        )
        fig.update_layout(title=f"Prediction Overlay: {SYMBOL} {TIMEFRAME} ({LABEL})")
        fig.show()


## 5. Backtest Artifact Review

If an evaluation report bundle exists, inspect the latest trades parquet and reuse the built-in MLFX report figures.

In [ ]:
trade_report_path = latest_trade_report_path(PATHS, SYMBOL, TIMEFRAME, LABEL)
trades_df = pl.read_parquet(trade_report_path) if trade_report_path else None

print(f"Trade artifact: {trade_report_path}")
if trades_df is None or trades_df.is_empty():
    print("Backtest section skipped. Run `pixi run mlfx evaluate ...` first.")
else:
    display(trades_df.head(5).to_pandas())
    show_or_note(get_equity_curve_figure(trades_df, f"Equity Curve: {SYMBOL} {TIMEFRAME} ({LABEL})"), "Could not render equity curve.")
    show_or_note(get_drawdown_analysis_figure(trades_df, f"Drawdown Analysis: {SYMBOL} {TIMEFRAME} ({LABEL})"), "Could not render drawdown analysis.")
    show_or_note(get_trade_distribution_figure(trades_df, f"Trade Distribution: {SYMBOL} {TIMEFRAME} ({LABEL})"), "Could not render trade distribution.")


## 6. Training Run History

Plot the training metrics log when it exists so you can inspect model iteration quality over time.

In [ ]:
run_log = metrics_log_path(PATHS, SYMBOL, TIMEFRAME, LABEL)
print(f"Metrics log: {run_log}")

if not run_log.exists():
    print("Run-history section skipped. Train at least one model first.")
else:
    records = [json.loads(line) for line in run_log.read_text().splitlines() if line.strip()]
    runs_df = pd.DataFrame(records)
    display(runs_df.tail(10))

    metric_columns = [col for col in ["best_cv_f1_macro", "accuracy", "elapsed_seconds"] if col in runs_df.columns]
    if metric_columns:
        fig = px.line(
            runs_df.reset_index(drop=True),
            x=runs_df.index,
            y=metric_columns,
            color_discrete_sequence=px.colors.qualitative.Bold,
            title="Training Metrics History",
        )
        fig.update_layout(xaxis_title="run index")
        fig.show()
    else:
        print("Metrics log exists but does not contain the expected numeric fields.")
